In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv("dataset_limpio.csv")


def _col(df: pd.DataFrame, j: str, part: str, axis: str) -> str | None:
    """Resuelve nombre de columna con variantes del dataset."""
    candidates = {
        "hombro": [f"{j}_hombro_d_{axis}", f"{j}_hombro_{axis}"],
        "codo": [f"{j}_codo_d_{axis}", f"{j}_codo_{axis}"],
        "muneca": [f"{j}_muneca_d_{axis}", f"{j}_muneca_{axis}"],
        "cabeza": [f"{j}_cabeza_{axis}"],
        "raqueta": [
            f"{j}_raqueta_{axis}",
            f"{j}_punta_raqueta_{axis}",
        ],
    }
    for name in candidates[part]:
        if name in df.columns:
            return name
    return None


def _missing_columns(df: pd.DataFrame, j: str) -> list[str]:
    needed = []
    for part in ("hombro", "codo", "muneca", "cabeza", "raqueta"):
        for axis in ("x", "y"):
            col = _col(df, j, part, axis)
            if col is None:
                needed.append(f"{j}_{part}_{axis}")
    vis = f"{j}_visible"
    if j != "j1" and vis not in df.columns:
        needed.append(vis)
    return needed


def angle_at_vertex(
    ax: np.ndarray,
    ay: np.ndarray,
    bx: np.ndarray,
    by: np.ndarray,
    cx: np.ndarray,
    cy: np.ndarray,
) -> np.ndarray:
    """Ángulo en B (vértice) entre los vectores B→A y B→C, en grados."""
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    m1 = np.hypot(v1x, v1y)
    m2 = np.hypot(v2x, v2y)
    denom = m1 * m2
    with np.errstate(invalid="ignore", divide="ignore"):
        cos_a = (v1x * v2x + v1y * v2y) / denom
    cos_a = np.clip(cos_a, -1.0, 1.0)
    out = np.degrees(np.arccos(cos_a))
    out[(denom == 0) | np.isnan(denom)] = np.nan
    return out


ANGLE_SPECS = [
    ("angulo_codo_d", "muneca", "codo", "hombro"),
    ("angulo_hombro_d", "codo", "hombro", "cabeza"),
    ("angulo_inclinacion_raqueta_d", "raqueta", "muneca", "codo"),
    ("angulo_raqueta_hombro_d", "raqueta", "muneca", "hombro"),
    ("angulo_muneca_cabeza_d", "muneca", "codo", "cabeza"),
]


def add_player_angles(df: pd.DataFrame, j: str) -> pd.DataFrame:
    missing = _missing_columns(df, j)
    if missing:
        print(f"[{j}] Columnas faltantes (no se calculan ángulos): {', '.join(missing)}")
        return df

    pts = {
        part: (df[_col(df, j, part, "x")].to_numpy(), df[_col(df, j, part, "y")].to_numpy())
        for part in ("hombro", "codo", "muneca", "cabeza", "raqueta")
    }

    if j == "j1":
        mask = np.ones(len(df), dtype=bool)
    else:
        mask = df[f"{j}_visible"].eq(1).to_numpy()

    for suffix, p1, vertex, p2 in ANGLE_SPECS:
        col_name = f"{j}_{suffix}"
        angles = angle_at_vertex(
            pts[p1][0], pts[p1][1],
            pts[vertex][0], pts[vertex][1],
            pts[p2][0], pts[p2][1],
        )
        angles = np.where(mask, angles, np.nan)
        df[col_name] = angles

    return df


for player in ("j1", "j2", "j3", "j4"):
    df = add_player_angles(df, player)

angle_cols = [c for c in df.columns if "_angulo_" in c]
df[angle_cols].head()

,cancha,partido,punto,frame,j1_confianza,j1_x_centro,j1_y_centro,j1_hombro_d_x,j1_hombro_d_y,j1_hombro_d_confianza,...,j4_cabeza_x,j4_cabeza_y,j4_cabeza_confianza,j1_raqueta_sospechosa,j2_raqueta_sospechosa,j3_raqueta_sospechosa,j4_raqueta_sospechosa,j2_visible,j3_visible,j4_visible
0,Cancha_1,Partido_1,Punto_10,0,0.887024,447.5,567.5,397.0,483.0,0.999765,...,NaN,NaN,NaN,False,False,False,False,0,0,0
1,Cancha_1,Partido_1,Punto_10,3,0.891826,442.5,570.0,395.0,486.0,0.999837,...,NaN,NaN,NaN,False,False,False,False,0,0,0
2,Cancha_1,Partido_1,Punto_10,6,0.880395,437.5,569.5,392.0,488.0,0.999471,...,NaN,NaN,NaN,False,False,False,False,1,0,0
3,Cancha_1,Partido_1,Punto_10,9,0.873153,428.0,568.0,399.0,489.0,0.999255,...,NaN,NaN,NaN,False,False,False,False,1,0,0
4,Cancha_1,Partido_1,Punto_10,12,0.870317,420.5,567.5,391.0,486.0,0.998572,...,NaN,NaN,NaN,False,False,False,False,0,0,0


In [ ]:
df.to_csv("datos_con_angulos.csv", index=False)
print(f"Guardado: datos_con_angulos.csv ({len(df):,} filas, {len(df.columns)} columnas)")

# Etiquetado de golpes (drive) por reglas de ángulos

A partir de **20 frames etiquetados manualmente** como drive, definimos rangos de ángulos
y etiquetamos el resto del dataset como **Y=1 (golpe)** o **Y=0 (no golpe)**.

Solo se evalúan jugadores visibles (`j1` siempre; `j2`–`j4` si `{j}_visible == 1`).

In [ ]:
# --- Cargar dataset y definir rangos de drive (20 frames confirmados) ---

df = pd.read_csv("datos_con_angulos.csv")

# Rangos observados en los 20 drives etiquetados (tabla_angulos_4_carpetas.csv)
DRIVE_RANGES = {
    "Codo": (93.37, 173.81),
    "Hombro": (128.20, 178.82),
    "Incl. Raqueta": (147.53, 180.00),
    "Raqueta-Hombro": (111.75, 176.19),
    "Muñeca-Cabeza": (89.22, 163.59),
}

# Mapeo nombre legible → columna en el dataset (por jugador j1..j4)
ANGLE_SUFFIX = {
    "Codo": "angulo_codo_d",
    "Hombro": "angulo_hombro_d",
    "Incl. Raqueta": "angulo_inclinacion_raqueta_d",
    "Raqueta-Hombro": "angulo_raqueta_hombro_d",
    "Muñeca-Cabeza": "angulo_muneca_cabeza_d",
}

PLAYERS = ["j1", "j2", "j3", "j4"]

def angle_col(player: str, angle_name: str) -> str:
    return f"{player}_{ANGLE_SUFFIX[angle_name]}"

def player_visible(df: pd.DataFrame, player: str) -> pd.Series:
    """j1: tiene detección; j2-j4: flag de visibilidad."""
    if player == "j1":
        return df["j1_confianza"].notna() & (df["j1_confianza"] > 0)
    return df[f"{player}_visible"].eq(1)

print(f"Dataset: {len(df):,} filas × {len(df.columns)} columnas")
print("\nRangos de drive (grados):")
for name, (lo, hi) in DRIVE_RANGES.items():
    print(f"  {name:18} [{lo:6.2f}°, {hi:6.2f}°]  (ancho = {hi - lo:.1f}°)")

In [ ]:
# --- ¿Qué ángulos discriminan mejor golpe vs no-golpe? ---
# Métrica: % de ángulos del dataset (jugadores visibles) que caen dentro del rango etiquetado.
# Un % alto = rango amplio / poco discriminativo. Un % bajo = mejor separación.

records = []
for player in PLAYERS:
    vis = player_visible(df, player)
    sub = df.loc[vis]
    for angle_name, (lo, hi) in DRIVE_RANGES.items():
        col = angle_col(player, angle_name)
        vals = sub[col].dropna()
        if vals.empty:
            continue
        in_range = vals.between(lo, hi).mean() * 100
        records.append({
            "jugador": player,
            "ángulo": angle_name,
            "n": len(vals),
            "min_dataset": vals.min(),
            "max_dataset": vals.max(),
            "ancho_rango_drive": hi - lo,
            "%_dentro_rango_drive": in_range,
        })

disc = pd.DataFrame(records)
disc_agg = (
    disc.groupby("ángulo", as_index=False)
    .agg(
        n=("n", "sum"),
        ancho_rango_drive=("ancho_rango_drive", "first"),
        pct_dentro_rango=("%_dentro_rango_drive", "mean"),
        min_dataset=("min_dataset", "min"),
        max_dataset=("max_dataset", "max"),
    )
    .sort_values("pct_dentro_rango")
)

print("=" * 72)
print("DISCRIMINACIÓN (menor % = más útil para filtrar)")
print("=" * 72)
for _, row in disc_agg.iterrows():
    tag = "★ útil" if row["pct_dentro_rango"] < 75 else ("⚠ débil" if row["pct_dentro_rango"] < 90 else "✗ poco útil")
    print(
        f"{row['ángulo']:18}  {row['pct_dentro_rango']:5.1f}% en rango drive  "
        f"(ancho drive {row['ancho_rango_drive']:.1f}°, dataset [{row['min_dataset']:.0f}°–{row['max_dataset']:.0f}°])  {tag}"
    )

print("\nConclusión:")
print("  • Más discriminativos: Muñeca-Cabeza, Raqueta-Hombro, Incl. Raqueta")
print("  • Poco útil para filtrar: Hombro (~96% del dataset ya cae en su rango)")
print("  • Codo: rango muy ancho (80°) → filtra poco por sí solo")

disc_agg

In [ ]:
# --- Análisis de margen recomendado (± grados) ---
# El margen compensa error de pose-estimation y la variabilidad de solo 20 ejemplos.

MARGENES = [0, 3, 5, 8, 10]
MARGEN_RECOMENDADO = 5  # ver justificación abajo

ANGULOS_DISCRIMINATIVOS = ["Muñeca-Cabeza", "Raqueta-Hombro", "Incl. Raqueta"]
ANGULOS_SIN_HOMBRO = [k for k in DRIVE_RANGES if k != "Hombro"]


def drive_mask_vectorized(sub: pd.DataFrame, player: str, margin: float, angle_names: list[str]) -> pd.Series:
    """Máscara booleana: True si todos los ángulos están en rango drive ± margen."""
    mask = pd.Series(True, index=sub.index)
    for angle_name in angle_names:
        lo, hi = DRIVE_RANGES[angle_name]
        col = angle_col(player, angle_name)
        mask &= sub[col].between(lo - margin, hi + margin)
    return mask


def pct_golpes(margin: float, angle_names: list[str]) -> tuple[int, int, float]:
    hits, total = 0, 0
    for player in PLAYERS:
        vis = player_visible(df, player)
        sub = df.loc[vis]
        total += len(sub)
        hits += drive_mask_vectorized(sub, player, margin, angle_names).sum()
    return hits, total, hits / total * 100 if total else 0.0


rows_margin = []
for m in MARGENES:
    for label, angles in [
        ("5 ángulos (todos)", list(DRIVE_RANGES.keys())),
        ("3 más discriminativos", ANGULOS_DISCRIMINATIVOS),
        ("sin Hombro", ANGULOS_SIN_HOMBRO),
    ]:
        hits, total, pct = pct_golpes(m, angles)
        rows_margin.append({
            "margen_±°": m,
            "criterio": label,
            "Y=1": hits,
            "evaluados": total,
            "%_Y=1": round(pct, 1),
        })

margin_df = pd.DataFrame(rows_margin)
print("Efecto del margen sobre jugadores visibles:\n")
print(margin_df.to_string(index=False))

print(f"\n>>> Margen recomendado: ±{MARGEN_RECOMENDADO}°")
print("    • ±0° es demasiado estricto (pose estimation ±2–5° de error típico)")
print("    • ±5° equilibra recall/precision con solo 20 drives de referencia")
print("    • ±10° marca ~70% como golpe → demasiados falsos positivos")
print("    • Excluir Hombro del filtro AND mejora poco (ya es muy permisivo)")

margin_df

In [ ]:
# --- Etiquetar Y=1 (golpe) / Y=0 (no golpe) por jugador ---

# Criterio: TODOS los 5 ángulos deben caer en rango drive ± margen (AND lógico).
# Si el jugador no es visible o falta algún ángulo → NaN (no evaluado).


def etiquetar_jugador(df_in: pd.DataFrame, margin: float = MARGEN_RECOMENDADO) -> pd.DataFrame:
    out = df_in.copy()
    angle_names = list(DRIVE_RANGES.keys())

    for player in PLAYERS:
        col_y = f"{player}_Y_golpe"
        vis = player_visible(out, player)
        out[col_y] = np.nan

        sub_idx = out.index[vis]
        sub = out.loc[sub_idx]
        angle_cols = [angle_col(player, a) for a in angle_names]
        complete = sub[angle_cols].notna().all(axis=1)

        if complete.any():
            ok = drive_mask_vectorized(sub.loc[complete], player, margin, angle_names)
            out.loc[ok[ok].index, col_y] = 1.0
            out.loc[ok[~ok].index, col_y] = 0.0

    return out


df = etiquetar_jugador(df, margin=MARGEN_RECOMENDADO)

# Resumen por jugador
print(f"Etiquetado con margen ±{MARGEN_RECOMENDADO}° (5 ángulos, criterio AND)\n")
for player in PLAYERS:
    col = f"{player}_Y_golpe"
    s = df[col]
    evaluados = s.notna().sum()
    pct = (s == 1).sum() / evaluados * 100 if evaluados else 0
    print(
        f"{player}:  Y=1 → {(s==1).sum():6,}  |  Y=0 → {(s==0).sum():6,}  |  "
        f"No evaluado → {s.isna().sum():6,}  |  % golpe sobre evaluados: {pct:.1f}%"
    )

# Columna a nivel frame: Y=1 si ALGÚN jugador visible es golpe
y_cols = [f"{p}_Y_golpe" for p in PLAYERS]
df["Y_golpe_frame"] = df[y_cols].max(axis=1, skipna=True)
df.loc[df[y_cols].isna().all(axis=1), "Y_golpe_frame"] = np.nan

print(f"\nA nivel frame (max jugadores): Y=1 → {(df['Y_golpe_frame']==1).sum():,}  |  Y=0 → {(df['Y_golpe_frame']==0).sum():,}")

df[[c for c in df.columns if "Y_golpe" in c]].head(10)

In [ ]:
# --- Guardar dataset etiquetado ---

OUT_CSV = "datos_con_angulos_etiquetados.csv"
df.to_csv(OUT_CSV, index=False)

print(f"Guardado: {OUT_CSV}")
print(f"  Filas: {len(df):,}")
print(f"  Columnas nuevas: {[c for c in df.columns if 'Y_golpe' in c]}")
print(f"  Margen usado: ±{MARGEN_RECOMENDADO}°")
print(f"  Rangos drive: {DRIVE_RANGES}")